# Convex Cost Experiment

Polars-based consolidation of each experiment under `logs/`, comparing the Longstaff-Schwartz Monte Carlo (LSM) benchmark price with the best-performing RL evaluation episode.

> Compatibility: validated inside the `EP11` conda environment (Python 3.11, NumPy 2.2.6, pandas 2.3.0) with `polars 1.35.1`.

In [1]:
from pathlib import Path
import polars as pl
import pandas as pd
from tqdm.notebook import tqdm

pd.options.display.float_format = "{:,.6f}".format

RISK_FREE_RATE = 0.05
MATURITY = 0.0833
N_RIGHTS = 22
DELTA_T = MATURITY / (N_RIGHTS - 1)


def locate_logs_dir(start_dir: Path | None = None) -> Path:
    start_dir = Path(start_dir or Path.cwd()).resolve()
    for current in (start_dir, *start_dir.parents):
        candidate = current / "logs"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Unable to find a 'logs' directory relative to this notebook.")


LOGS_DIR = locate_logs_dir()
LOGS_DIR

PosixPath('/Users/alexanderithakis/Documents/GitHub/DRL-Swing-Options/logs')

In [2]:
discount_expr = (-RISK_FREE_RATE * DELTA_T * pl.col("time_step")).exp()


def compute_lsm_price(lsm_csv: Path) -> float:
    df = pl.read_csv(lsm_csv)
    discounted = df.with_columns(
        (pl.col("payoff") * discount_expr).alias("discounted_payoff")
    )
    path_totals = (
        discounted.group_by("path")
        .agg(pl.col("discounted_payoff").sum())
        .select(pl.col("discounted_payoff"))
    )
    return float(path_totals.mean().item())


def compute_best_rl_price(rl_csv_files: list[Path]) -> tuple[float | None, str | None]:
    best_price = None
    best_episode = None
    for rl_csv in rl_csv_files:
        df = pl.read_csv(rl_csv)
        episode_price = (
            df.group_by("path")
            .agg(pl.col("reward").sum().alias("reward_sum"))
            .select(pl.col("reward_sum").mean())
            .item()
        )
        if episode_price is None:
            continue
        if best_price is None or episode_price > best_price:
            best_price = float(episode_price)
            best_episode = rl_csv.stem
    return best_price, best_episode

In [3]:
experiment_rows: list[dict[str, object]] = []
experiment_dirs = sorted(LOGS_DIR.iterdir())
for experiment_dir in tqdm(experiment_dirs, total=len(experiment_dirs)):
    eval_dir = experiment_dir / "evaluations"
    lsm_file = eval_dir / "lsm.csv"
    rl_files = sorted(eval_dir.glob("rl_episode_*.csv"))
    if not eval_dir.is_dir() or not lsm_file.exists() or not rl_files:
        continue

    lsm_price = compute_lsm_price(lsm_file)
    best_rl_price, best_episode = compute_best_rl_price(rl_files)

    experiment_rows.append(
        {
            "name": experiment_dir.name,
            "LSM": lsm_price,
            "Best RL": best_rl_price,
            "best_episode": best_episode,
        }
    )

summary_df = pl.DataFrame(experiment_rows).sort("name")
summary_view = summary_df.select(["name", "LSM", "Best RL"])
summary_view.to_pandas()

  0%|          | 0/120 [00:00<?, ?it/s]

,name,LSM,Best RL
0,SwingOption_20_11,2.625979,2.588738
1,SwingOption_20_12,2.684629,2.538439
2,SwingOption_20_13,2.651677,2.608920
3,SwingOption_20_14,2.684180,2.660009
4,SwingOption_20_c0.01_gamma1.5_11,2.460233,2.441792
...,...,...,...
115,SwingOption_20_gamma2_c0.1_14,1.014534,1.368484
116,SwingOption_20_gamma3_c0.8_11,-3.061989,0.458465
117,SwingOption_20_gamma3_c0.8_12,-3.026449,0.462088
118,SwingOption_20_gamma3_c0.8_13,-3.083657,0.465761


In [4]:
sumdf = summary_view.to_pandas()

sumdf['PctDiff'] = (sumdf.loc[:,'Best RL'] - sumdf.loc[:,'LSM']) / sumdf.loc[:,'LSM'].abs() * 100
sumdf.set_index('name', inplace=True)
sumdf.to_csv('Convex Costs Results.csv')
sumdf

,LSM,Best RL,PctDiff
name,,,
SwingOption_20_11,2.625979,2.588738,-1.418151
SwingOption_20_12,2.684629,2.538439,-5.445450
SwingOption_20_13,2.651677,2.608920,-1.612475
SwingOption_20_14,2.684180,2.660009,-0.900494
SwingOption_20_c0.01_gamma1.5_11,2.460233,2.441792,-0.749565
...,...,...,...
SwingOption_20_gamma2_c0.1_14,1.014534,1.368484,34.887886
SwingOption_20_gamma3_c0.8_11,-3.061989,0.458465,114.972787
SwingOption_20_gamma3_c0.8_12,-3.026449,0.462088,115.268306


In [26]:
# make a 3x3 pandas table with x asis c and y axis gamma, I am going to fill the values

# name,LSM,Best RL,PctDiff

# SwingOption_20_c0.15_gamma2_11,0.48435276209974243,0.9993280021972656,106.32235023601963
# SwingOption_20_gamma1.5_c0.3_11,0.09925343649705881,0.46132950219726565,364.79952581887306
# SwingOption_20_gamma2_c0.1_14,1.0145340884554201,1.368483587890625,34.88788631775558

# c: 0.01, 0.02, 0.04, 0.05, 0.08, 0.10, 0.15
data = {'1'  : [-0.3738528953998654, -0.23894159842134335, None, -0.8736946954098761, None, None, None], # c: 0.01, 0.02, 0.05
        '1.5': [None, -0.6203779299347628, 0.3820217053196209, None, 3.541400433811437, None, None], # c: 0.02, 0.04, 0.08
        '2'  : [None, None, None, None, 6.661315752888408, 35.703079791772055, 106.32235023601963], # c: 0.05, 0.10, 0.15
        }
df = pd.DataFrame.from_dict(data).T
df.index.name = 'gamma'
df.columns = [0.01, 0.02, 0.04, 0.05, 0.08, 0.10, 0.15]
df.columns.name = 'c'
df

c,0.010000,0.020000,0.040000,0.050000,0.080000,0.100000,0.150000
gamma,,,,,,,
1,-0.373853,-0.238942,NaN,-0.873695,NaN,NaN,NaN
1.5,NaN,-0.620378,0.382022,NaN,3.541400,NaN,NaN
2,NaN,NaN,NaN,NaN,6.661316,35.703080,106.322350
